In [1]:
"""
g++ -O3 -Wall -shared -std=c++14 -fPIC $(python3 -m pybind11 --includes) "/mnt/c/Users/Aneesh Shastri/Documents/GitHub/EpochCoreTasks/Task1/Subtask2/Custom_Tokenizers/bpe_tokenizer.cpp" -o"/mnt/c/Users/Aneesh Shastri/Documents/GitHub/EpochCoreTasks/Task1/Subtask2/Custom_Tokenizers/bpe_tokenizer" $(python3-config --extension-suffix)
"""

'\ng++ -O3 -Wall -shared -std=c++14 -fPIC $(python3 -m pybind11 --includes) "/mnt/c/Users/Aneesh Shastri/Documents/GitHub/EpochCoreTasks/Task1/Subtask2/Custom_Tokenizers/bpe_tokenizer.cpp" -o"/mnt/c/Users/Aneesh Shastri/Documents/GitHub/EpochCoreTasks/Task1/Subtask2/Custom_Tokenizers/bpe_tokenizer" $(python3-config --extension-suffix)\n'

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from flax import nnx
import optax
import grain.python as grain
import bpe_tokenizer

DATASET_PATH="./Code_Dataset/train.parquet"
RANDOM_SEED=14

In [3]:

df = pd.read_parquet(
    "./Code_Dataset/train.parquet",
    engine="pyarrow",
    dtype_backend="pyarrow"
)
corpus=df['buggy'].str.cat(sep='\u0000')+df['fixed'].str.cat(sep='\u0000')


In [4]:

Tokenizer = bpe_tokenizer.BPETokenizer()
Tokenizer.train(corpus, num_merges=1000)
VOCAB_SIZE = Tokenizer.get_vocab_size()
print(f"Vocab size: {VOCAB_SIZE}")

# Calculate the max sequence length across buggy and fixed inputs
max_buggy_len = max(len(Tokenizer.encode(x)) for x in df['buggy'])
max_fixed_len = max(len(Tokenizer.encode(x)) for x in df['fixed'])
MAX_SEQ_LEN = max(max_buggy_len, max_fixed_len)
print(f"Max sequence length: {MAX_SEQ_LEN}")

Vocab size: 1256
Max sequence length: 261


In [5]:
class TextSource(grain.RandomAccessDataSource):
    def __init__(self,bugged,fixed,tokenizer,max_len=MAX_SEQ_LEN):
        self.X=[]
        self.y=[]
        pad_token = tokenizer.get_vocab_size()
        for i in range(len(bugged)):
            input_chunk=tokenizer.encode(bugged[i])
            target_chunk=tokenizer.encode(fixed[i])
            
            if len(input_chunk) < max_len:
                input_chunk = input_chunk + [pad_token] * (max_len - len(input_chunk))
            else:
                input_chunk = input_chunk[:max_len]
                
            if len(target_chunk) < max_len:
                target_chunk = target_chunk + [pad_token] * (max_len - len(target_chunk))
            else:
                target_chunk = target_chunk[:max_len]
                
            self.X.append(input_chunk)
            self.y.append(target_chunk)
        
        self.X = jnp.array(self.X, dtype=jnp.int32)
        self.y = jnp.array(self.y, dtype=jnp.int32)
           

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {"input": self.X[idx], "output": self.y[idx]}

In [6]:
"""enc_sample=Tokenizer.encode(corpus)
print(len(enc_sample))
enc_sample=enc_sample[50:]
context_size = 4        
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(Tokenizer.decode(context), "---->", Tokenizer.decode([desired]))"""

'enc_sample=Tokenizer.encode(corpus)\nprint(len(enc_sample))\nenc_sample=enc_sample[50:]\ncontext_size = 4        \nx = enc_sample[:context_size]\ny = enc_sample[1:context_size+1]\nprint(f"x: {x}")\nprint(f"y:      {y}")\n\nfor i in range(1, context_size+1):\n    context = enc_sample[:i]\n    desired = enc_sample[i]\n    print(Tokenizer.decode(context), "---->", Tokenizer.decode([desired]))'

In [7]:
def make_loader(bugged,fixed,tokenizer=Tokenizer, max_len=MAX_SEQ_LEN, stride=128, batch_size=4, training=True, seed=RANDOM_SEED):
    source = TextSource(bugged,fixed,tokenizer,max_len)
    transforms = [grain.Batch(batch_size, drop_remainder=True)]

    return grain.DataLoader(
        data_source=source,
        sampler=grain.IndexSampler(
            num_records=len(source),
            shuffle=training,
            seed=seed,
            num_epochs=1,
            shard_options=grain.NoSharding(),
        ),
        worker_count=6,
        worker_buffer_size=2,
        operations=transforms,
    )

In [8]:
class RoPE(nnx.Module):
    def __init__(self, head_dim: int, max_seq_len: int = 4096, base: float = 10000.0):
        if head_dim % 2 != 0:
            raise ValueError(f"head_dim must be even. Got {head_dim}")
            
        # Compute angles
        inv_freq = 1.0 / (base ** (jnp.arange(0, head_dim, 2) / head_dim))
        positions = jnp.arange(max_seq_len)
        angles = jnp.outer(positions, inv_freq)
        
        # Cache complex exponentials e^(i * theta)
        complex_freqs = jnp.exp(1j * angles)
        self.complex_freqs = nnx.Cache(complex_freqs)

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        Expects x to have shape: (batch_size, seq_len, n_heads, head_dim)
        """
        seq_len = x.shape[1]
        freqs = self.complex_freqs[...][:seq_len, :]
        
        # 2. Reshape for broadcasting: (1, seq_len, 1, head_dim // 2)
        freqs = freqs[None, :, None, :]
        
        # 3. Group head_dim into pairs: (..., head_dim // 2, 2)
        x_paired = x.reshape(*x.shape[:-1], -1, 2)
        
        # 4. Map to complex plane and multiply
        x_complex = jax.lax.complex(x_paired[..., 0], x_paired[..., 1])
        rotated_complex = x_complex * freqs
        
        # 5. Map back to real coordinates and flatten
        rotated_real = jnp.stack([rotated_complex.real, rotated_complex.imag], axis=-1)
        return rotated_real.reshape(x.shape)

In [9]:
class FeedforwardNN(nnx.Module):
    def __init__(self, d_model: int, hidden_dim: int,rngs: nnx.Rngs):
        self.w_gate= nnx.Linear(d_model, hidden_dim, use_bias=False, rngs=rngs)
        self.w_up= nnx.Linear(d_model, hidden_dim, use_bias=False, rngs=rngs)
        self.w_down= nnx.Linear(hidden_dim,d_model, use_bias=False, rngs=rngs)
    def __call__(self, x:jax.Array):
        hidden_state=self.w_up(x)*jax.nn.silu(self.w_gate(x))
        return self.w_down(hidden_state)

In [10]:
class MHSA(nnx.Module):
    def __init__(self, d_model,max_seq_len, d_k=None, n_heads=8,rngs=None):
        
        self.d_model = d_model
        self.n_heads=n_heads
        if d_k is not None: self.d_k=d_k
        else: self.d_k=d_model
        self.head_size=self.d_k//n_heads
        self.max_seq=max_seq_len
        # Linear layers for projecting input X into Q, K, V spaces
        self.w_q = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_k = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_v = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_o = nnx.Linear(self.d_k, d_model, use_bias=False, rngs=rngs)
        self.RoPE=RoPE(head_dim=self.head_size,max_seq_len=max_seq_len)
        
    def __call__(self, x: jax.Array, mask: jax.Array | None = None) -> jax.Array:
        """
        x shape: (batch_size, seq_len, d_model)
        """
        batch,seq_len ,_ = x.shape
        # 1. Project inputs
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)
        Q=q.reshape((batch,seq_len,self.n_heads,self.head_size))
        K=k.reshape((batch,seq_len,self.n_heads,self.head_size))
        V=v.reshape((batch,seq_len,self.n_heads,self.head_size))
        #apply RoPE
        Q_R=self.RoPE(Q)
        K_R=self.RoPE(K)

        #Permute matrix for computing Attention tensor
        Q_R = jnp.transpose(Q_R, (0, 2, 1, 3))
        K_R = jnp.transpose(K_R, (0, 2, 1, 3))
        V = jnp.transpose(V, (0, 2, 1, 3))
        
        # Scaled dot-product scores: (batch, n_heads, seq_len, seq_len)
        scale = 1.0 / jnp.sqrt(self.head_size)
        attn_scores = jnp.matmul(Q_R, jnp.transpose(K_R, (0, 1, 3, 2))) * scale
        if mask is not None:
            attn_scores = jnp.where(mask, attn_scores, -1e12)
        attn_weights = jax.nn.softmax(attn_scores, axis=-1)
        self.sown_attn = nnx.Intermediate(attn_weights)
        # Weighted sum over values: (batch, n_heads, seq_len, head_size)
        context = jnp.matmul(attn_weights, V)
        
        # Permute back to (batch, seq_len, n_heads, head_size) and flatten heads
        context = jnp.transpose(context, (0, 2, 1, 3))
        attn_output = context.reshape((batch, seq_len, self.d_k))
        
        # Final projection
        return self.w_o(attn_output)

In [11]:
class MHCA(nnx.Module):
    def __init__(self, d_model, max_seq_len, d_k=None, n_heads=8, rngs=None):
        self.d_model = d_model
        self.n_heads = n_heads
        if d_k is not None: self.d_k = d_k
        else: self.d_k = d_model
        self.head_size = self.d_k // n_heads
        self.max_seq = max_seq_len
        # Linear layers for projecting input X (queries) and Y (keys/values) into Q, K, V spaces
        self.w_q = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_k = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_v = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_o = nnx.Linear(self.d_k, d_model, use_bias=False, rngs=rngs)
        self.RoPE = RoPE(head_dim=self.head_size, max_seq_len=max_seq_len)
        
    def __call__(self, x: jax.Array, y: jax.Array, mask: jax.Array | None = None) -> jax.Array:
        """
        x shape (query): (batch_size, seq_len_q, d_model)
        y shape (key/value): (batch_size, seq_len_kv, d_model)
        """
        batch, seq_len_q, _ = x.shape
        _, seq_len_kv, _ = y.shape
        # 1. Project inputs
        q = self.w_q(x)
        k = self.w_k(y)
        v = self.w_v(y)
        Q = q.reshape((batch, seq_len_q, self.n_heads, self.head_size))
        K = k.reshape((batch, seq_len_kv, self.n_heads, self.head_size))
        V = v.reshape((batch, seq_len_kv, self.n_heads, self.head_size))
        # apply RoPE
        Q_R = self.RoPE(Q)
        K_R = self.RoPE(K)

        # Permute matrix for computing Attention tensor
        Q_R = jnp.transpose(Q_R, (0, 2, 1, 3))
        K_R = jnp.transpose(K_R, (0, 2, 1, 3))
        V = jnp.transpose(V, (0, 2, 1, 3))
        
        # Scaled dot-product scores: (batch, n_heads, seq_len_q, seq_len_kv)
        scale = 1.0 / jnp.sqrt(self.head_size)
        attn_scores = jnp.matmul(Q_R, jnp.transpose(K_R, (0, 1, 3, 2))) * scale
        if mask is not None:
            attn_scores = jnp.where(mask, attn_scores, -1e12)
        attn_weights = jax.nn.softmax(attn_scores, axis=-1)
        self.sown_attn = nnx.Intermediate(attn_weights)
        # Weighted sum over values: (batch, n_heads, seq_len_q, head_size)
        context = jnp.matmul(attn_weights, V)
        
        # Permute back to (batch, seq_len_q, n_heads, head_size) and flatten heads
        context = jnp.transpose(context, (0, 2, 1, 3))
        attn_output = context.reshape((batch, seq_len_q, self.d_k))
        
        # Final projection
        return self.w_o(attn_output)

In [12]:
# Verification code for MHCA
key = jax.random.PRNGKey(0)
rngs = nnx.Rngs(0)
# Create dummy queries (batch=2, seq_len_q=10, d_model=16)
x_dummy = jax.random.normal(key, (2, 10, 16))
# Create dummy keys/values (batch=2, seq_len_kv=15, d_model=16)
y_dummy = jax.random.normal(key, (2, 15, 16))

mhca = MHCA(d_model=16, max_seq_len=20, n_heads=4, rngs=rngs)
out_dummy = mhca(x_dummy, y_dummy)
print("MHCA forward pass output shape:", out_dummy.shape)
assert out_dummy.shape == (2, 10, 16), f"Expected shape (2, 10, 16), got {out_dummy.shape}"
print("MHCA verification test passed successfully!")


MHCA forward pass output shape: (2, 10, 16)
MHCA verification test passed successfully!


In [13]:


class TransformerBlock(nnx.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, rngs: nnx.Rngs):
        self.attn_norm = nnx.RMSNorm(d_model, rngs=rngs)
        self.mha = MHSA(d_model, n_heads=n_heads, max_seq_len=max_seq_len, rngs=rngs)
        
        self.ffn_norm = nnx.RMSNorm(d_model, rngs=rngs)
        hidden_dim = int((8/3) * d_model) 
        self.ffn = FeedForwardNN(d_model, hidden_dim, rngs)

    def __call__(self, x: jax.Array, mask: jax.Array | None = None) -> jax.Array:
        """
        x shape: (batch_size, seq_len, d_model)
        """
        # Phase 1: Pre-Norm Attention Residual Stream
        x_norm1 = self.attn_norm(x)
        attn_out = self.mha(x_norm1, mask)
        x = x + attn_out  # First residual addition
        
        # Phase 2: Pre-Norm FeedForward Residual Stream
        x_norm2 = self.ffn_norm(x)
        ffn_out = self.ffn(x_norm2)
        x = x + ffn_out  # Second residual addition
        
        return x

In [14]:
# Load validation and test datasets
val_df = pd.read_parquet(
    "./Code_Dataset/validation.parquet",
    engine="pyarrow",
    dtype_backend="pyarrow"
)
test_df = pd.read_parquet(
    "./Code_Dataset/test.parquet",
    engine="pyarrow",
    dtype_backend="pyarrow"
)

# Create train, validation, and test dataloaders
BATCH_SIZE = 512
train_loader = make_loader(df['buggy'], df['fixed'], batch_size=BATCH_SIZE, training=True)
val_loader = make_loader(val_df['buggy'], val_df['fixed'], batch_size=BATCH_SIZE, training=False)
test_loader = make_loader(test_df['buggy'], test_df['fixed'], batch_size=BATCH_SIZE, training=False)

print("Dataloaders successfully created.")


Dataloaders successfully created.


In [15]:
# RNN + Self-attention Model
class RNNSelfAttentionModel(nnx.Module):
    def __init__(self, vocab_size: int, d_model: int = 128, hidden_dim: int = 128, n_heads: int = 4, max_seq_len: int = MAX_SEQ_LEN, rngs: nnx.Rngs = None):
        self.embed = nnx.Embed(vocab_size, d_model, rngs=rngs)
        # Bidirectional LSTM to capture context from both directions
        fw = nnx.RNN(nnx.LSTMCell(d_model, hidden_dim // 2, rngs=rngs))
        bw = nnx.RNN(nnx.LSTMCell(d_model, hidden_dim // 2, rngs=rngs))
        self.lstm = nnx.Bidirectional(fw, bw)
        # Multi-head self-attention on top of LSTM hidden states
        self.mha = MHSA(d_model=hidden_dim, max_seq_len=max_seq_len, n_heads=n_heads, rngs=rngs)
        self.head = nnx.Linear(hidden_dim, vocab_size, rngs=rngs)
        self.pad_token = vocab_size - 1
        
    def __call__(self, x: jax.Array) -> jax.Array:
        mask = (x != self.pad_token)[:, None, None, :]
        h = self.embed(x)      # (batch, seq_len, d_model)
        h = self.lstm(h)       # (batch, seq_len, hidden_dim)
        h = self.mha(h, mask=mask)        # (batch, seq_len, hidden_dim)
        logits = self.head(h)  # (batch, seq_len, vocab_size)
        return logits


In [16]:
# Loss, train_step, validation_step
def compute_loss(logits, targets, pad_token=None):
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets.astype(jnp.int32))
    if pad_token is not None:
        mask = (targets != pad_token).astype(jnp.float32)
        loss = (loss * mask).sum() / (mask.sum() + 1e-8)
    else:
        loss = loss.mean()
    return loss

@nnx.jit
def train_step(model, optimizer, batch, pad_token):
    X, y = batch['input'], batch['output']
    def loss_fn(model):
        logits = model(X)
        loss = compute_loss(logits, y, pad_token)
        return loss, logits
    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model, grads)
    return loss, logits

@nnx.jit
def validation_step(model, batch, pad_token):
    X, y = batch['input'], batch['output']
    logits = model(X)
    loss = compute_loss(logits, y, pad_token)
    return loss, logits


In [17]:
# Evaluation and EarlyStopping utilities
def evaluate(model, dataloader, pad_token):
    total_tokens = 0
    correct_tokens = 0
    total_seqs = 0
    correct_seqs = 0
    
    for batch in dataloader:
        X, y = batch['input'], batch['output']
        logits = model(X)
        preds = jnp.argmax(logits, axis=-1)
        
        mask = (y != pad_token)
        correct_tokens += np.sum((preds == y) & mask)
        total_tokens += np.sum(mask)
        
        # A sequence is correct if all non-pad tokens match
        seq_match = np.all((preds == y) | (~mask), axis=1)
        correct_seqs += np.sum(seq_match)
        total_seqs += len(y)
        
    token_acc = correct_tokens / (total_tokens + 1e-8)
    seq_acc = correct_seqs / (total_seqs + 1e-8)
    return float(token_acc), float(seq_acc)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None
    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = jax.tree.map(lambda x: x, nnx.state(model))
        else:
            self.counter += 1
        return self.counter >= self.patience


In [ ]:
# Model, Optimizer, and Training loop
import time

rngs = nnx.Rngs(params=0, dropout=1)
TOTAL_VOCAB_SIZE = VOCAB_SIZE + 1
pad_token = VOCAB_SIZE

# Model initialization
d_model = 128
hidden_dim = 128
n_heads = 4

model = RNNSelfAttentionModel(
    vocab_size=TOTAL_VOCAB_SIZE,
    d_model=d_model,
    hidden_dim=hidden_dim,
    n_heads=n_heads,
    max_seq_len=MAX_SEQ_LEN,
    rngs=rngs
)

# Scheduler & Optimizer
lr = 1e-3
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=lr,
    warmup_steps=100,
    decay_steps=2000,
    end_value=1e-5
)

optimizer = nnx.Optimizer(
    model,
    optax.adamw(schedule, weight_decay=1e-4),
    wrt=nnx.Param
)

es = EarlyStopping(patience=5)
epochs = 15

print("Starting training of RNN+Self-Attention Model...")
for epoch in range(epochs):
    t0 = time.time()
    train_losses = []
    for batch in train_loader:
        loss, _ = train_step(model, optimizer, batch, pad_token)
        train_losses.append(loss)
        #print(f"Batch {batch} finished.")
    val_losses = []
    for batch in val_loader:
        val_loss, _ = validation_step(model, batch, pad_token)
        val_losses.append(val_loss)
        
    mean_train_loss = np.mean(train_losses)
    mean_val_loss = np.mean(val_losses)
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {mean_train_loss:.4f} | Val Loss: {mean_val_loss:.4f} | Time: {time.time()-t0:.2f}s")
    
    if es.step(mean_val_loss, model):
        print("Early stopping triggered. Restoring best model state...")
        break

if es.best_state is not None:
    nnx.update(model, es.best_state)
    print("Restored best model parameters.")


Starting training of RNN+Self-Attention Model...


In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_token_acc, test_seq_acc = evaluate(model, test_loader, pad_token)
print(f"Test Token Accuracy: {test_token_acc:.4f}")
print(f"Test Exact Sequence Accuracy: {test_seq_acc:.4f}")
